# 🧪 Exploratory Data Analysis (EDA)

Notebook ini berisi proses **Exploratory Data Analysis (EDA)** terhadap dataset rata-rata harga pangan bulanan tingkat konsumen nasional. Tahap ini dilakukan untuk memahami karakteristik data sebelum proses preprocessing dan pemodelan machine learning.

Analisis yang dilakukan meliputi:
- Pemeriksaan struktur dan informasi dataset.
- Analisis statistik deskriptif.
- Pemeriksaan missing value dan data duplikat.
- Analisis jumlah data berdasarkan komoditas, tahun, dan bulan.
- Visualisasi distribusi harga komoditas.
- Analisis tren harga berdasarkan waktu.
- Visualisasi hubungan antarvariabel sebagai dasar proses preprocessing dan pemodelan.

In [ ]:
#  import
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

In [ ]:
# LOAD DATASET


DATA_PATH = "../data/raw/1778217759.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# Display basic information about the DataFrame, including data types and non-null values
df.info()

In [ ]:
# Display descriptive statistics
display(df.describe(include="all"))

In [ ]:
# Mengecek dataset menggunakan isnull().sum()
missing_awal = df.isnull().sum().reset_index()
missing_awal.columns = ["Kolom", "Jumlah_Missing"]

display(missing_awal)

print("Total missing value awal:", df.isnull().sum().sum())

In [ ]:
print("Jumlah duplikasi seluruh baris:", df.duplicated().sum())

In [ ]:
print("Nama kolom:")
print(df.columns.tolist())

In [ ]:
print("Jumlah komoditas unik:", df["Komoditas"].nunique())

display(df["Komoditas"].value_counts())

In [ ]:
print("Daftar bulan:")
print(df["Bulan"].unique())

display(df["Bulan"].value_counts())

In [ ]:
df_eda = df.copy()

df_eda.columns = df_eda.columns.str.strip()

df_eda["Komoditas"] = df_eda["Komoditas"].astype(str).str.strip()
df_eda["Bulan"] = df_eda["Bulan"].astype(str).str.strip()

df_eda["Harga"] = (
    df_eda["Harga"]
    .astype(str)
    .str.replace("Rp", "", regex=False)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.replace("-", "", regex=False)
    .str.strip()
)

df_eda["Harga"] = df_eda["Harga"].replace("", np.nan)
df_eda["Harga"] = pd.to_numeric(df_eda["Harga"], errors="coerce")

display(df_eda.head())

###Visualisasi Jumlah Data Per Komoditas

In [ ]:
jumlah_komoditas = df_eda["Komoditas"].value_counts().sort_values(ascending=True)

plt.figure(figsize=(12, 10))
plt.barh(jumlah_komoditas.index, jumlah_komoditas.values)
plt.title("Jumlah Data per Komoditas")
plt.xlabel("Jumlah Data")
plt.ylabel("Komoditas")
plt.tight_layout()
plt.show()

###Visualisasi Jumlah Data Per Tahun

In [ ]:
jumlah_tahun = df_eda["Tahun"].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(jumlah_tahun.index.astype(str), jumlah_tahun.values)
plt.title("Jumlah Data per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Jumlah Data")
plt.tight_layout()
plt.show()

###Visualisasi Jumlah Data Per Bulan

In [ ]:
urutan_bulan = [
    "Januari", "Februari", "Maret", "April", "Mei", "Juni",
    "Juli", "Agustus", "September", "Oktober", "November", "Desember"
]

jumlah_bulan = df_eda["Bulan"].value_counts().reindex(urutan_bulan)

plt.figure(figsize=(12, 5))
plt.bar(jumlah_bulan.index, jumlah_bulan.values)
plt.title("Jumlah Data per Bulan")
plt.xlabel("Bulan")
plt.ylabel("Jumlah Data")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

###Visualisasi Distribusi Harga Komoditas

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df_eda["Harga"].dropna(), bins=30)
plt.title("Distribusi Harga Komoditas")
plt.xlabel("Harga")
plt.ylabel("Frekuensi")
plt.tight_layout()
plt.show()

###Boxplot Harga Komoditas

In [ ]:
plt.figure(figsize=(10, 5))
plt.boxplot(df_eda["Harga"].dropna(), vert=False)
plt.title("Boxplot Harga Komoditas")
plt.xlabel("Harga")
plt.tight_layout()
plt.show()

###Visualisasi Rata-Rata Harga Per Komoditas

In [ ]:
rata_harga_komoditas = (
    df_eda.groupby("Komoditas")["Harga"]
    .mean()
    .sort_values(ascending=True)
)

plt.figure(figsize=(12, 10))
plt.barh(rata_harga_komoditas.index, rata_harga_komoditas.values)
plt.title("Rata-rata Harga per Komoditas")
plt.xlabel("Rata-rata Harga")
plt.ylabel("Komoditas")
plt.tight_layout()
plt.show()

###Visualisasi Tren Rata-Rata Harga Per Tahun

In [ ]:
tren_tahunan = df_eda.groupby("Tahun")["Harga"].mean()

plt.figure(figsize=(10, 5))
plt.plot(tren_tahunan.index, tren_tahunan.values, marker="o")
plt.title("Tren Rata-rata Harga per Tahun")
plt.xlabel("Tahun")
plt.ylabel("Rata-rata Harga")
plt.grid(True)
plt.tight_layout()
plt.show()

###Visualisasi Tren Harga Beberapa Komoditas

In [ ]:
bulan_mapping_eda = {
    "Januari": 1,
    "Februari": 2,
    "Maret": 3,
    "April": 4,
    "Mei": 5,
    "Juni": 6,
    "Juli": 7,
    "Agustus": 8,
    "September": 9,
    "Oktober": 10,
    "November": 11,
    "Desember": 12
}

df_eda["Bulan_Num"] = df_eda["Bulan"].map(bulan_mapping_eda)

df_eda["Tanggal"] = pd.to_datetime(
    df_eda["Tahun"].astype(str) + "-" + df_eda["Bulan_Num"].astype("Int64").astype(str) + "-01",
    errors="coerce"
)

top_komoditas = df_eda["Komoditas"].value_counts().head(5).index.tolist()

plt.figure(figsize=(14, 7))

for komoditas in top_komoditas:
    subset = df_eda[df_eda["Komoditas"] == komoditas].sort_values("Tanggal")
    plt.plot(subset["Tanggal"], subset["Harga"], marker="o", label=komoditas)

plt.title("Tren Harga Beberapa Komoditas")
plt.xlabel("Tanggal")
plt.ylabel("Harga")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

###Heatmap Rata-rata Harga per Tahun dan Bulan

In [ ]:
pivot_harga = df_eda.pivot_table(
    values="Harga",
    index="Tahun",
    columns="Bulan",
    aggfunc="mean"
)

pivot_harga = pivot_harga.reindex(columns=urutan_bulan)

plt.figure(figsize=(14, 6))
plt.imshow(pivot_harga, aspect="auto")
plt.colorbar(label="Rata-rata Harga")
plt.title("Heatmap Rata-rata Harga per Tahun dan Bulan")
plt.xlabel("Bulan")
plt.ylabel("Tahun")
plt.xticks(range(len(pivot_harga.columns)), pivot_harga.columns, rotation=45)
plt.yticks(range(len(pivot_harga.index)), pivot_harga.index)
plt.tight_layout()
plt.show()